In [1]:
import json
import os
from getpass import getpass
from dotenv import load_dotenv

from haystack.document_stores.in_memory import InMemoryDocumentStore
from haystack import Document

from haystack import Pipeline
from haystack.components.embedders import SentenceTransformersDocumentEmbedder, SentenceTransformersTextEmbedder
from haystack.components.writers import DocumentWriter
from haystack.document_stores.in_memory import InMemoryDocumentStore
from haystack.document_stores.types import DuplicatePolicy

from haystack.components.builders import AnswerBuilder, PromptBuilder
from haystack.components.generators import OpenAIGenerator
from haystack.components.retrievers.in_memory import InMemoryEmbeddingRetriever

/Users/patrickcallery/dev/pydata-manchester-talk/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()

True

In [3]:
NATURAL_QUESTIONS_FILE_PATH = "../data/subset_training_data.jsonl"

In [4]:
wiki_docs = []
with open(NATURAL_QUESTIONS_FILE_PATH, 'r') as file:
    for i, line in enumerate(file):
        wiki_docs.append(json.loads(line))

In [5]:
first_10_docs = wiki_docs[0:10]

In [6]:
example_wiki_document = first_10_docs[0].get("document_text")

In [7]:
example_wiki_document

"Email marketing - Wikipedia <H1> Email marketing </H1> Jump to : navigation , search <Table> <Tr> <Td> </Td> <Td> ( hide ) This article has multiple issues . Please help improve it or discuss these issues on the talk page . ( Learn how and when to remove these template messages ) <Table> <Tr> <Td> </Td> <Td> This article needs additional citations for verification . Please help improve this article by adding citations to reliable sources . Unsourced material may be challenged and removed . ( September 2014 ) ( Learn how and when to remove this template message ) </Td> </Tr> </Table> <Table> <Tr> <Td> </Td> <Td> This article possibly contains original research . Please improve it by verifying the claims made and adding inline citations . Statements consisting only of original research should be removed . ( January 2015 ) ( Learn how and when to remove this template message ) </Td> </Tr> </Table> ( Learn how and when to remove this template message ) </Td> </Tr> </Table> <Table> <Tr> <T

In [8]:
document_store = InMemoryDocumentStore()

In [9]:
docs_for_vector_db = [Document(content=doc["document_text"][0:10000]) for doc in first_10_docs]

In [10]:
document_embedder = SentenceTransformersDocumentEmbedder(model="sentence-transformers/all-MiniLM-L6-v2")
document_writer = DocumentWriter(document_store=document_store, policy=DuplicatePolicy.SKIP)

In [11]:
indexing = Pipeline()
indexing.add_component(instance=document_embedder, name="document_embedder")
indexing.add_component(instance=document_writer, name="document_writer")

indexing.connect("document_embedder.documents", "document_writer.documents")

indexing.run({"document_embedder": {"documents": docs_for_vector_db}})

Batches: 100%|██████████| 1/1 [00:02<00:00,  2.19s/it]


{'document_writer': {'documents_written': 10}}

In [12]:
if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("Enter OpenAI API key:")

In [13]:
template = """
        You have to answer the following question based on the given context information only.

        Context:
        {% for document in documents %}
            {{ document.content }}
        {% endfor %}

        Question: {{question}}
        Answer:
        """

rag_pipeline = Pipeline()
rag_pipeline.add_component(
    "query_embedder", SentenceTransformersTextEmbedder(model="sentence-transformers/all-MiniLM-L6-v2")
)
rag_pipeline.add_component("retriever", InMemoryEmbeddingRetriever(document_store, top_k=3))
rag_pipeline.add_component("prompt_builder", PromptBuilder(template=template))
rag_pipeline.add_component("generator", OpenAIGenerator(model="gpt-3.5-turbo"))
rag_pipeline.add_component("answer_builder", AnswerBuilder())

rag_pipeline.connect("query_embedder", "retriever.query_embedding")
rag_pipeline.connect("retriever", "prompt_builder.documents")
rag_pipeline.connect("prompt_builder", "generator")
rag_pipeline.connect("generator.replies", "answer_builder.replies")
rag_pipeline.connect("generator.meta", "answer_builder.meta")
rag_pipeline.connect("retriever", "answer_builder.documents")

🚅 Components
  - query_embedder: SentenceTransformersTextEmbedder
  - retriever: InMemoryEmbeddingRetriever
  - prompt_builder: PromptBuilder
  - generator: OpenAIGenerator
  - answer_builder: AnswerBuilder
🛤️ Connections
  - query_embedder.embedding -> retriever.query_embedding (List[float])
  - retriever.documents -> prompt_builder.documents (List[Document])
  - retriever.documents -> answer_builder.documents (List[Document])
  - prompt_builder.prompt -> generator.prompt (str)
  - generator.replies -> answer_builder.replies (List[str])
  - generator.meta -> answer_builder.meta (List[Dict[str, Any]])

In [14]:
first_10_docs[1]

{'document_text': 'The Mother ( How I Met Your Mother ) - wikipedia <H1> The Mother ( How I Met Your Mother ) </H1> Jump to : navigation , search <Table> <Tr> <Th_colspan="2"> Tracy McConnell </Th> </Tr> <Tr> <Td_colspan="2"> How I Met Your Mother character </Td> </Tr> <Tr> <Td_colspan="2"> The Mother appearing in `` The Locket \'\' </Td> </Tr> <Tr> <Th> First appearance </Th> <Td> `` Lucky Penny ( unseen ) \'\' `` Something New \'\' ( seen ) </Td> </Tr> <Tr> <Th> Last appearance </Th> <Td> `` Last Forever \'\' </Td> </Tr> <Tr> <Th> Created by </Th> <Td> Carter Bays Craig Thomas </Td> </Tr> <Tr> <Th> Portrayed by </Th> <Td> Cristin Milioti </Td> </Tr> <Tr> <Th_colspan="2"> Information </Th> </Tr> <Tr> <Th> Aliases </Th> <Td> The Mother </Td> </Tr> <Tr> <Th> Gender </Th> <Td> Female </Td> </Tr> <Tr> <Th> Spouse ( s ) </Th> <Td> Ted Mosby </Td> </Tr> <Tr> <Th> Significant other ( s ) </Th> <Td> Max ( deceased former boyfriend ) Louis ( ex-boyfriend ) </Td> </Tr> <Tr> <Th> Children </Th> 

In [19]:
question = "Name of the song which Robyn sang in the Canadian pop band?"

response = rag_pipeline.run(
    {
        "query_embedder": {"text": question},
        "prompt_builder": {"question": question},
        "answer_builder": {"query": question},
    }
)
print(response["answer_builder"]["answers"][0].data)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00,  2.44it/s]


The song that Robin sang in the Canadian pop band was "Let's Go to the Mall."
